In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import root_mean_squared_error as rmse
import time

import math
import matplotlib as mpl
from scipy.spatial import ConvexHull
import os
# our functions
import predict_Beta_I
import choice_start_day
import plot_hyb

import warnings
warnings.filterwarnings(action='ignore')

# to account for updates when files change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from langchain_core.runnables import RunnableLambda, RunnableSequence

runnable1 = RunnableLambda(lambda x: x + 1)
runnable2 = RunnableLambda(lambda x: x + 2)

#создаем цепочку из двух обьект RunnableLambda
chain = RunnableSequence(runnable1, runnable2)

#вызывваем цепочку
print(chain.invoke(2)) 

5


In [8]:
from langchain_core.prompts import PromptTemplate, FewShotChatMessagePromptTemplate

prompt_template = PromptTemplate.from_template("Tell me a joke about {topic}")
print(prompt_template.invoke({"topic": "cats"}))

text='Tell me a joke about cats'


In [9]:
auth_key = 'MDE5YmM2MjctOWU5ZS03OWE0LWE0OGYtY2FhMTU4MzYxZDgzOjJmNWUxMzMzLTNlYzMtNGI1Ni04OGZiLTRhMGIxNzFjNzFkMw=='

In [40]:
def get_token(auth_token, scope='GIGACHAT_API_PERS'):
    """
      Выполняет POST-запрос к эндпоинту, который выдает токен.

      Параметры:
      - auth_token (str): токен авторизации, необходимый для запроса.
      - область (str): область действия запроса API. По умолчанию — «GIGACHAT_API_PERS».

      Возвращает:
      - ответ API, где токен и срок его "годности".
      """
    # Создадим идентификатор UUID (36 знаков)
    rq_uid = str(uuid.uuid4())

    # API URL
    url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"

    # Заголовки
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Accept': 'application/json',
        'RqUID': rq_uid,
        'Authorization': f'Basic {auth_token}'
    }

    # Тело запроса
    payload = {
        'scope': scope
    }

    try:
        # Делаем POST запрос с отключенной SSL верификацией
        # (можно скачать сертификаты Минцифры, тогда отключать проверку не надо)
        response = requests.post(url, headers=headers, data=payload, verify=False)
        return response
    except requests.RequestException as e:
        print(f"Ошибка: {str(e)}")
        return -1


In [11]:
import requests

url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"

payload={
  'scope': 'GIGACHAT_API_PERS'
}
headers = {
  'Content-Type': 'application/x-www-form-urlencoded',
  'Accept': 'application/json',
  'RqUID': '1b44ce11-da28-43e3-b665-3d44c3c469f9',
  'Authorization': f'Basic MDE5YmM2MjctOWU5ZS03OWE0LWE0OGYtY2FhMTU4MzYxZDgzOjJmNWUxMzMzLTNlYzMtNGI1Ni04OGZiLTRhMGIxNzFjNzFkMw=='
}

response = requests.request("POST", url, headers=headers, data=payload, verify=False)

print(response.json)

{"access_token":"eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.GdXr0WltYK8LFvMFIW4OeFg6FO7zlM1wGAVvKMpBuXZy4lhl_0fH0hxi4MvTIH3yo2xFtDj5JkesBVwAvCHqil2IOdv2aSaR_dH21Epmle5tjATNzvcy4C--ifkZeGOXX_rpsIu6A0hPe5CsLGbqeSkIrTGHBrsJ_NNZEju27aAayhMjG06CukF6PIYghrq_T4ZuW4Z8c7J-CjZN2fGU_7CGLE-I4ur-3hEUVqwbAKFSg0r34ay-KcPMBSQDDRn2--OxiEsdJ_ED8coEx3LZhw9udyQAvSbMHU_cUq67UoyzUuUdZ8ojm2srHIjhV1WTzP50Jmq9SioOvUzwQQdehQ.Om2bvLDP7swRZowKiQih4g.58teGwJ0jiXVoVypuuBQH8WtHzsN87r_YJOhky1es7IWCprLFAxEiorBCauaw8jOpDuw8TdFTY3bxTKYbK3MKNYAEu_zytLqZT3ahh58WGhzLabf0xi34sW_0_8TN-Bvf1hfeU4W6-fXzj8NjdWEb5bC9LBuEUpSVMA5s26pJurb20i0c04vNs5MOOcOkF6DRrDIRnrEwF_Q2WHVePmfUzYITJHlt_rtLKHpJwExXDmMCLp8ZCROir6eVvPOrsOVVcmy_Sy19XUrJEbBaHteGeqbUqyRkbBP15pP7E7WNXauwq2n0oodwHfmocLn-GMTcbrZjAfGLwMKF3T8kH99oFltaWnZctUowbVigAQSBdvGNDwzW7ykcPGg_FjlbCBjkeRd3p4GxDSmJESnu7eHw6zGmFMUpzHVKSKSQ9YII_BT6aRFlQxRm0XU2S-FREQveQGfDLEcLgpL19dQ1JndjF7iYKNQOQDyj816j0uCfiQ6hrVrfyBCGxs0eYlkDmfNGIF97a20c5vjoDtSpmdK-a3OK9OiK

In [21]:
auth_key

'MDE5YmM2MjctOWU5ZS03OWE0LWE0OGYtY2FhMTU4MzYxZDgzOjJmNWUxMzMzLTNlYzMtNGI1Ni04OGZiLTRhMGIxNzFjNzFkMw=='

In [12]:
access_token="eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.GdXr0WltYK8LFvMFIW4OeFg6FO7zlM1wGAVvKMpBuXZy4lhl_0fH0hxi4MvTIH3yo2xFtDj5JkesBVwAvCHqil2IOdv2aSaR_dH21Epmle5tjATNzvcy4C--ifkZeGOXX_rpsIu6A0hPe5CsLGbqeSkIrTGHBrsJ_NNZEju27aAayhMjG06CukF6PIYghrq_T4ZuW4Z8c7J-CjZN2fGU_7CGLE-I4ur-3hEUVqwbAKFSg0r34ay-KcPMBSQDDRn2--OxiEsdJ_ED8coEx3LZhw9udyQAvSbMHU_cUq67UoyzUuUdZ8ojm2srHIjhV1WTzP50Jmq9SioOvUzwQQdehQ.Om2bvLDP7swRZowKiQih4g.58teGwJ0jiXVoVypuuBQH8WtHzsN87r_YJOhky1es7IWCprLFAxEiorBCauaw8jOpDuw8TdFTY3bxTKYbK3MKNYAEu_zytLqZT3ahh58WGhzLabf0xi34sW_0_8TN-Bvf1hfeU4W6-fXzj8NjdWEb5bC9LBuEUpSVMA5s26pJurb20i0c04vNs5MOOcOkF6DRrDIRnrEwF_Q2WHVePmfUzYITJHlt_rtLKHpJwExXDmMCLp8ZCROir6eVvPOrsOVVcmy_Sy19XUrJEbBaHteGeqbUqyRkbBP15pP7E7WNXauwq2n0oodwHfmocLn-GMTcbrZjAfGLwMKF3T8kH99oFltaWnZctUowbVigAQSBdvGNDwzW7ykcPGg_FjlbCBjkeRd3p4GxDSmJESnu7eHw6zGmFMUpzHVKSKSQ9YII_BT6aRFlQxRm0XU2S-FREQveQGfDLEcLgpL19dQ1JndjF7iYKNQOQDyj816j0uCfiQ6hrVrfyBCGxs0eYlkDmfNGIF97a20c5vjoDtSpmdK-a3OK9OiKdNZD7jyO0bCTg2Y3DybawTmGI-KyUY4N2bZOvkhxKVW_aYaWypQ1LWp-GHZhZ1wI5c5rEXsRsUqrSC9xSh2qvauT5E4Ke6bfeTh4LDX-OPxw8RrcaiU_lfqxGdFXZA502OzbTIA91tXSTw6_94SuCkK0kcrwqfksU4HRt7arlRqxNdYZ3rsgeerXAV7c05_S65aBoRJXw6IKts.StB-SYVH63CUAO_EsuRZzp9X4uELR8yLngh1tl2fBcc"

In [57]:
response = get_token(auth_key)
if response != 1:
    print(response.text)
    access_token = response.json()['access_token']

{"access_token":"eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.nbIMaw6bDaJTCmrUl97Zli3MFJRnDDpUpgctDUWok0ETMinondpwwfizzGUqR1J8faezHkEa804-NJscMWkkoiU65UfElzws25196hhxmRb0qrMtcwhZC-jNa7tiaqubIMm-Ae1aw9V7RXcQ5iKkAVLPuVilnPSPOFvBZfYqrupgLxqiXrTlzSEDLiHWTx9PljyaTxxH6ZNKYvI6mYpQ3Ps2Ow2IgCuUeBPmGrXF8w_DxM14iq0dIxAWGEtrlxAs3v82Q4K1SKE1Ux8absQjo1A5pp0QtpM--43bLFihW6Zzc97J2bH1pZRXyZw8GX-fnDnMCyRqe099p62dlB2BCw.IPBvjz2xKbYYX4MJPuwEkw.aIww0LFCPsXrHxyZ_2jYlqxWnTheFg8EnKkxP986zEgxg68uqcr01brsb_hEy1lJ8g0HbQFQG5hcOqjE4FPCQ_GkZ6w1cc6VmB5yiqhYw2K72Ow7d_W9amdBBrqoXmooGhJ6vFBp69SyFBU-SciAINZ3NsQZAx9R9fZQRvr1r9P7GLMGbcJnrnI1ySzIj73S_fhgre6XTcl0stGoiw0_Lqgq4nnVEPXP1sgX4MeSTS2303-hoyy9gxYERwiNkD-Y2lV6NfiYljsWW66Js6v-VLad6DUdt8cPfC5S5rM-HE5VfltQjr9FFt63BznufP9MipYmklif2r7WpfnfyERLKPhRJSbIIPd_J1at8tWTNiHJzFyqu37ZXHgyTKIl7wAx60LuSWwAIY7cW4IRgrGAdD2JoTNfX5N-awaqheqbr5dalS9NXsfofKveEoCRjj6s7fAfsKzabd9HJI8gQIkuWM3RzKbIgY06gkdDe2YeBu0h8H5voBbpH6K9vfBVHiGAn4_zeaE4bA2xxB5EUkPJcGcZCF7B2

In [58]:
access_token

'eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.nbIMaw6bDaJTCmrUl97Zli3MFJRnDDpUpgctDUWok0ETMinondpwwfizzGUqR1J8faezHkEa804-NJscMWkkoiU65UfElzws25196hhxmRb0qrMtcwhZC-jNa7tiaqubIMm-Ae1aw9V7RXcQ5iKkAVLPuVilnPSPOFvBZfYqrupgLxqiXrTlzSEDLiHWTx9PljyaTxxH6ZNKYvI6mYpQ3Ps2Ow2IgCuUeBPmGrXF8w_DxM14iq0dIxAWGEtrlxAs3v82Q4K1SKE1Ux8absQjo1A5pp0QtpM--43bLFihW6Zzc97J2bH1pZRXyZw8GX-fnDnMCyRqe099p62dlB2BCw.IPBvjz2xKbYYX4MJPuwEkw.aIww0LFCPsXrHxyZ_2jYlqxWnTheFg8EnKkxP986zEgxg68uqcr01brsb_hEy1lJ8g0HbQFQG5hcOqjE4FPCQ_GkZ6w1cc6VmB5yiqhYw2K72Ow7d_W9amdBBrqoXmooGhJ6vFBp69SyFBU-SciAINZ3NsQZAx9R9fZQRvr1r9P7GLMGbcJnrnI1ySzIj73S_fhgre6XTcl0stGoiw0_Lqgq4nnVEPXP1sgX4MeSTS2303-hoyy9gxYERwiNkD-Y2lV6NfiYljsWW66Js6v-VLad6DUdt8cPfC5S5rM-HE5VfltQjr9FFt63BznufP9MipYmklif2r7WpfnfyERLKPhRJSbIIPd_J1at8tWTNiHJzFyqu37ZXHgyTKIl7wAx60LuSWwAIY7cW4IRgrGAdD2JoTNfX5N-awaqheqbr5dalS9NXsfofKveEoCRjj6s7fAfsKzabd9HJI8gQIkuWM3RzKbIgY06gkdDe2YeBu0h8H5voBbpH6K9vfBVHiGAn4_zeaE4bA2xxB5EUkPJcGcZCF7B2I67TjhDiaHlWP4AB

In [24]:
from langchain_gigachat.chat_models import GigaChat

giga = GigaChat(
    # Для авторизации запросов используйте ключ, полученный в проекте GigaChat API
    credentials=auth_key,
    verify_ssl_certs=False,
)

print(giga.invoke("Hello, world!"))

ConfigError: unable to infer type for attribute "x_headers"

In [27]:
import requests

url = "https://gigachat.devices.sberbank.ru/api/v1/models"

payload = {}
headers = {
  'Accept': 'application/json',
  'Authorization': f'Bearer {access_token}'
}

response = requests.request("GET", url, headers=headers, data=payload, verify=False)

response.json()

{'object': 'list',
 'data': [{'id': 'GigaChat',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-Max',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-Pro',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Max',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Max-preview',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Plus',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Pro',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Pro-preview',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-preview',
   'object': 'model',
   'owned_by': '

In [45]:
import json

def get_chat_completion(auth_token, user_message):
    """
    Отправляет POST-запрос к API чата для получения ответа от модели GigaChat.

    Параметры:
    - auth_token (str): Токен для авторизации в API.
    - user_message (str): Сообщение от пользователя, для которого нужно получить ответ.

    Возвращает:
    - str: Ответ от API в виде текстовой строки.
    """
    # URL API, к которому мы обращаемся
    url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

    # Подготовка данных запроса в формате JSON
    payload = json.dumps({
        "model": "GigaChat",  # Используемая модель
        "messages": [
            {
                "role": "user",  # Роль отправителя (пользователь)
                "content": user_message  # Содержание сообщения
            }
        ],
        "temperature": 1,  # Температура генерации
        "top_p": 0.1,  # Параметр top_p для контроля разнообразия ответов
        "n": 1,  # Количество возвращаемых ответов
        "stream": False,  # Потоковая ли передача ответов
        "max_tokens": 50,  # Максимальное количество токенов в ответе
        "repetition_penalty": 1,  # Штраф за повторения
        "update_interval": 0  # Интервал обновления (для потоковой передачи)
    })

    # Заголовки запроса
    headers = {
        'Content-Type': 'application/json',  # Тип содержимого - JSON
        'Accept': 'application/json',  # Принимаем ответ в формате JSON
        'Authorization': f'Bearer {auth_token}'  # Токен авторизации
    }

    # Выполнение POST-запроса и возвращение ответа
    try:
        response = requests.request("POST", url, headers=headers, data=payload, verify=False)
        return response
    except requests.RequestException as e:
        # Обработка исключения в случае ошибки запроса
        print(f"Произошла ошибка: {str(e)}")
        return -1


In [46]:
answer = get_chat_completion(access_token, 'Как на Пайтоне сделать GET запрос?')
answer.json()

{'choices': [{'message': {'content': 'На Python для выполнения GET-запроса чаще всего используют библиотеку `requests`, которая проста в использовании и удобна для большинства задач.\n\nВот простой пример выполнения GET-запроса:\n\n```python\nimport requests\n\n# URL, на который отправ',
    'role': 'assistant'},
   'index': 0,
   'finish_reason': 'length'}],
 'created': 1768560792,
 'model': 'GigaChat:2.0.28.2',
 'object': 'chat.completion',
 'usage': {'prompt_tokens': 18,
  'completion_tokens': 50,
  'total_tokens': 68,
  'precached_prompt_tokens': 2}}

In [48]:
def get_chat_completion(auth_token, user_message, conversation_history=None):
    """
    Отправляет POST-запрос к API чата для получения ответа от модели GigaChat в рамках диалога.

    Параметры:
    - auth_token (str): Токен для авторизации в API.
    - user_message (str): Сообщение от пользователя, для которого нужно получить ответ.
    - conversation_history (list): История диалога в виде списка сообщений (опционально).

    Возвращает:
    - response (requests.Response): Ответ от API.
    - conversation_history (list): Обновленная история диалога.
    """
    # URL API, к которому мы обращаемся
    url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

    # Если история диалога не предоставлена, инициализируем пустым списком
    if conversation_history is None:
        conversation_history = []

    # Добавляем сообщение пользователя в историю диалога
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Подготовка данных запроса в формате JSON
    payload = json.dumps({
        "model": "GigaChat:latest",
        "messages": conversation_history,
        "temperature": 1,
        "top_p": 0.1,
        "n": 1,
        "stream": False,
        "max_tokens": 512,
        "repetition_penalty": 1,
        "update_interval": 0
    })

    # Заголовки запроса
    headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/json',
        'Authorization': f'Bearer {auth_token}'
    }

    # Выполнение POST-запроса и возвращение ответа
    try:
        response = requests.post(url, headers=headers, data=payload, verify=False)
        response_data = response.json()
        print(response_data)

        # Добавляем ответ модели в историю диалога
        conversation_history.append({
            "role": "assistant",
            "content": response_data['choices'][0]['message']['content']
        })

        return response, conversation_history
    except requests.RequestException as e:
        # Обработка исключения в случае ошибки запроса
        print(f"Произошла ошибка: {str(e)}")
        return None, conversation_history


In [51]:
# Пример использования функции для диалога

conversation_history = []

# Пользователь отправляет первое сообщение
response, conversation_history = get_chat_completion(access_token, "Привет, как дела?", conversation_history)

# Пользователь отправляет следующее сообщение, продолжая диалог
response, conversation_history = get_chat_completion(access_token, "Что ты умеешь делать?", conversation_history)

{'choices': [{'message': {'content': 'Привет! Всё отлично, готов общаться и помогать. Как твои дела?', 'role': 'assistant'}, 'index': 0, 'finish_reason': 'stop'}], 'created': 1768561246, 'model': 'GigaChat:2.0.28.2', 'object': 'chat.completion', 'usage': {'prompt_tokens': 16, 'completion_tokens': 16, 'total_tokens': 32, 'precached_prompt_tokens': 2}}
{'choices': [{'message': {'content': 'Могу помочь с множеством задач:\n\n- Разобраться в сложных темах и объяснить простыми словами.\n- Решать задачи по математике, физике, химии и другим предметам.\n- Писать тексты, статьи, сценарии, стихи и рассказы.\n- Помочь с программированием, написанием кода и отладкой.\n- Обсуждать философию, науку, культуру и другие темы.\n- Давать советы по саморазвитию, психологии и личностному росту.\n- Помогать с анализом информации, созданием презентаций и отчетов.\n- Поддержать беседу на любую тему.\n\nЕсли нужно что-то конкретное — спрашивай!', 'role': 'assistant'}, 'index': 0, 'finish_reason': 'stop'}], 'c

In [52]:
conversation_history

[{'role': 'user', 'content': 'Привет, как дела?'},
 {'role': 'assistant',
  'content': 'Привет! Всё отлично, готов общаться и помогать. Как твои дела?'},
 {'role': 'user', 'content': 'Что ты умеешь делать?'},
 {'role': 'assistant',
  'content': 'Могу помочь с множеством задач:\n\n- Разобраться в сложных темах и объяснить простыми словами.\n- Решать задачи по математике, физике, химии и другим предметам.\n- Писать тексты, статьи, сценарии, стихи и рассказы.\n- Помочь с программированием, написанием кода и отладкой.\n- Обсуждать философию, науку, культуру и другие темы.\n- Давать советы по саморазвитию, психологии и личностному росту.\n- Помогать с анализом информации, созданием презентаций и отчетов.\n- Поддержать беседу на любую тему.\n\nЕсли нужно что-то конкретное — спрашивай!'}]

In [60]:
access_token, auth_key

('eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.nbIMaw6bDaJTCmrUl97Zli3MFJRnDDpUpgctDUWok0ETMinondpwwfizzGUqR1J8faezHkEa804-NJscMWkkoiU65UfElzws25196hhxmRb0qrMtcwhZC-jNa7tiaqubIMm-Ae1aw9V7RXcQ5iKkAVLPuVilnPSPOFvBZfYqrupgLxqiXrTlzSEDLiHWTx9PljyaTxxH6ZNKYvI6mYpQ3Ps2Ow2IgCuUeBPmGrXF8w_DxM14iq0dIxAWGEtrlxAs3v82Q4K1SKE1Ux8absQjo1A5pp0QtpM--43bLFihW6Zzc97J2bH1pZRXyZw8GX-fnDnMCyRqe099p62dlB2BCw.IPBvjz2xKbYYX4MJPuwEkw.aIww0LFCPsXrHxyZ_2jYlqxWnTheFg8EnKkxP986zEgxg68uqcr01brsb_hEy1lJ8g0HbQFQG5hcOqjE4FPCQ_GkZ6w1cc6VmB5yiqhYw2K72Ow7d_W9amdBBrqoXmooGhJ6vFBp69SyFBU-SciAINZ3NsQZAx9R9fZQRvr1r9P7GLMGbcJnrnI1ySzIj73S_fhgre6XTcl0stGoiw0_Lqgq4nnVEPXP1sgX4MeSTS2303-hoyy9gxYERwiNkD-Y2lV6NfiYljsWW66Js6v-VLad6DUdt8cPfC5S5rM-HE5VfltQjr9FFt63BznufP9MipYmklif2r7WpfnfyERLKPhRJSbIIPd_J1at8tWTNiHJzFyqu37ZXHgyTKIl7wAx60LuSWwAIY7cW4IRgrGAdD2JoTNfX5N-awaqheqbr5dalS9NXsfofKveEoCRjj6s7fAfsKzabd9HJI8gQIkuWM3RzKbIgY06gkdDe2YeBu0h8H5voBbpH6K9vfBVHiGAn4_zeaE4bA2xxB5EUkPJcGcZCF7B2I67TjhDiaHlWP4A

In [69]:
from langchain_gigachat.chat_models import GigaChat

giga = GigaChat(
    # Для авторизации запросов используйте ключ, полученный в проекте GigaChat API
    credentials=auth_key,
    verify_ssl_certs=False,
)

print(giga.invoke("Hello, world!"))

ConfigError: unable to infer type for attribute "x_headers"

In [70]:
from langchain_gigachat.chat_models import GigaChat
giga = GigaChat(
    # Для авторизации запросов используйте ключ, полученный в проекте GigaChat API
    credentials=auth_key,
    verify_ssl_certs=False,
)

print(giga.invoke("Hello, world!"))

ConfigError: unable to infer type for attribute "x_headers"

In [67]:
giga = GigaChat(credentials=auth_key,
                model='GigaChat',
                # пока название модели такое,
                # обещают, что скоро можно будет просто GigaChat указывать
                verify_ssl_certs=False,
                scope='GIGACHAT_API_PERS'
                )

ConfigError: unable to infer type for attribute "x_headers"

In [33]:
import requests
import json

url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

payload = json.dumps({
  "model": "GigaChat",
  "messages": [
    {
      "role": "user",
      "content": "Погода в Болхове",
      "functions_state_id": "string",
      "attachments": [
        "string"
      ]
    }
  ],
  "function_call": "none",
  "functions": [
    {
      "name": "weather_forecast",
      "description": "Прогноз погоды",
      "parameters": {
        "properties": {
          "location": {
            "type": "string",
            "description": "Местоположение, например, название города"
          }
        }
      },
      "few_shot_examples": [
        {
          "request": "Погода в Москве в ближайшие три дня",
          "params": {
            "location": "Moscow, Russia"
          }
        }
      ],
      "return_parameters": {
        "properties": {
          "location": {
            "type": "string",
            "description": "Местоположение, например, название города"
          }
        }
      }
    }
  ],
  "temperature": 0,
  "top_p": 0,
  "stream": False,
  "max_tokens": 30,
  "repetition_penalty": 1,
  "update_interval": 0
})
headers = {
  'Content-Type': 'application/json',
  'Accept': 'application/json',
  'Authorization': f'Bearer {access_token}'
}

response = requests.request("POST", url, headers=headers, data=payload, verify=False)

print(response.text)

{"status":401,"message":"Token has expired"}



In [31]:
import uuid

In [32]:
rq_uid = str(uuid.uuid4())

'38a211bf-0671-4747-8109-37da24f81d69'

In [30]:
from gigachat import GigaChat

giga = GigaChat(
    credentials=auth_key,
    verify_ssl_certs=False,
)

response = giga.get_models()

print(response)

ConfigError: unable to infer type for attribute "x_headers"